# JP-1 先頭1回の確認（変更 A-2・GPU・Sarashina2.2 1B・float32）

G1 診断で、float32 なら G1 が通ることはわかりました。ただ、32 本の続きを作るときに同じ先頭を 32 回計算しているせいで、作業2つが L4 に収まりません。変更 A-2 では、**先頭を1行で1回だけ計算して、32 本に配ります**。このノートブックはその確認用で、計画のゲートではありません。

- **作業2つを同時に**（本番と同じく別々のプロセス）、本番でいちばん長くなる長さ（いちばん長い頼み方＋512 トークン）で回して、メモリが足りるかを見ます。
- **新しいやり方と古いやり方**で、同じ乱数から同じ 32 本ができるかを比べます。E2・G1・B1 も確かめます。
- G0 のセル7 と同じ式で、**1冊の時間と CU** を出し直します。
- 頼み方の長さを測るために MGSM の問題を読み込みますが、数（トークンの数）だけを使います。画面とファイルに出すのは数・真偽・ハッシュだけです。
- 不合格でも止まらず最後まで回ります。止まるのは版が合わないときだけです。

**使い方**：ランタイムは **GPU（L4）**。シークレットは `GITHUB_TOKEN`。「すべてのセルを実行」で **15〜20 分くらい**。結果は Drive の `DECISION_POINTS/jp1/PREFIXCHECK/`。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。

In [ ]:
# Cell 1 — Drive, GPU, exact head, notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'JP-1 先頭1回の確認（変更A-2・GPU・Sarashina 1B・float32・作業2つのメモリ）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - JP-1 prefix check', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'jp1_prefixcheck'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), "GPU のランタイム（L4）にしてね"
GPU_NAME = gpu.stdout.strip().split(",")[0]
EXPECTED_HEAD = "b9516df5d07500d9919ad4959d6505d4db954d31"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
token = None
env.pop("GITHUB_TOKEN", None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
D = Path("/content/drive/MyDrive/DECISION_POINTS/jp1/PREFIXCHECK")
D.mkdir(parents=True, exist_ok=True)
JST = dt.timezone(dt.timedelta(hours=9))
STARTED = time.time()
print("GPU:", GPU_NAME, "| head:", head)

In [ ]:
# Cell 2 — pinned packages, versions, G0 run 1's numbers, and the longest prompt (tokens only).
from pork import jp1 as P
PINS = [f"{k}=={v}" for k, v in P.PACKAGES.items()] + ["protobuf"]
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-3000:]
import importlib.metadata as md
CHECKS = {"packages": {k: md.version(k) for k in P.PACKAGES}, "torch": md.version("torch"),
          "gate_doc_sha256_ok": P.gate_doc_sha256(REPO) == P.GATE_DOC_SHA256}
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
SPEC = P.MODELS["1b"]
MODEL_DIR = Path("/content/sarashina_1b")
snapshot_download(SPEC["id"], revision=SPEC["revision"], local_dir=str(MODEL_DIR), token=False,
                  allow_patterns=["*.json", "*.safetensors", "tokenizer*", "*.model"])
TOK = AutoTokenizer.from_pretrained(str(MODEL_DIR))
_vocab = sorted(TOK.get_vocab().items())
CHECKS["tokenizer_1b"] = {
    "vocab_sha256_ok": hashlib.sha256(json.dumps(_vocab, ensure_ascii=False).encode("utf-8")).hexdigest() == P.TOKENIZER_VOCAB_SHA256,
    "chat_template_sha256_ok": hashlib.sha256((TOK.chat_template or "").encode("utf-8")).hexdigest() == P.CHAT_TEMPLATE_SHA256}
del _vocab
G0D = Path("/content/drive/MyDrive/DECISION_POINTS/jp1/G0")
_g0 = json.loads((G0D / "G0_ACCURACY.json").read_text(encoding="utf-8"))
assert _g0["chosen"] == "1b", "G0 の1回目で選んだモデルが 1B ではありません — Claude に見せてね"
MEDIAN = {L: int(_g0["1b"][L]["median_tokens"]) for L in ("ja", "en")}
del _g0
_g0c = json.loads((G0D / "G0_CHECKS.json").read_text(encoding="utf-8"))["checks"]
from pork import kv_cached_backend as kv
PROMPT_LEN = {}
for L in ("ja", "en"):
    _rows = P.load_rows(L)
    _main, _g0i = P.items(_rows, "main"), P.items(_rows, "g0")
    CHECKS[f"items_same_as_g0_run1_{L}"] = (P.items_sha256(_main) == _g0c["items_sha256"][L]
                                           and P.items_sha256(_g0i) == _g0c["g0_items_sha256"][L])
    _lens = sorted(len(kv.chat_prompt_ids(TOK, P.user_message(L, it["question"]))) for it in _main + _g0i)
    PROMPT_LEN[L] = {"max": _lens[-1], "median": _lens[len(_lens) // 2], "n": len(_lens)}
    del _rows, _main, _g0i, _lens
LONGEST = max(v["max"] for v in PROMPT_LEN.values()) + P.MAX_NEW_TOKENS     # prompt + 512: the longest sequence in the run
VERSION_OK = (CHECKS["gate_doc_sha256_ok"] and CHECKS["packages"] == P.PACKAGES
              and CHECKS["torch"].split("+")[0] == P.TORCH_BASE_VERSION and all(CHECKS["tokenizer_1b"].values())
              and CHECKS["items_same_as_g0_run1_ja"] and CHECKS["items_same_as_g0_run1_en"])
def write_json(name, obj):
    p = D / name
    p.write_text(json.dumps(obj, ensure_ascii=False, indent=1, sort_keys=True, default=str), encoding="utf-8")
    return hashlib.sha256(p.read_bytes()).hexdigest()
write_json("PREFIX_CHECKS.json", {"head": EXPECTED_HEAD, "checks": CHECKS, "version_ok": VERSION_OK, "gpu": GPU_NAME,
                                  "prompt_tokens": PROMPT_LEN, "longest_sequence": LONGEST, "g0_run1_median_base_tokens": MEDIAN})
print(json.dumps({"version_ok": VERSION_OK, "prompt_tokens": PROMPT_LEN, "longest_sequence": LONGEST}, indent=1))
assert VERSION_OK, P.STOP_VERSION + " — 推論の前に止めました。Claude に見せてね。"
print("VERSIONS OK")

In [ ]:
# Cell 3 — two worker processes at once (as the main run), float32, the new step, the longest sequence.
PROBE_PY = Path("/content/jp1_probe.py")
PROBE_PY.write_text('\nimport json, sys, time, traceback\nfrom pathlib import Path\nrepo, model_dir, length, steps, out, tag, barrier = sys.argv[1:8]\nlength, steps = int(length), int(steps)\nsys.path.insert(0, repo)\nres = {"tag": tag, "ok": False}\ndef wait(name, n=2, timeout=300):\n    Path(barrier, name + "_" + tag).touch()\n    t0 = time.time()\n    while len(list(Path(barrier).glob(name + "_*"))) < n:\n        if time.time() - t0 > timeout:\n            raise SystemExit("barrier timeout: " + name)\n        time.sleep(0.5)\ntry:\n    from pork import jp1_runner as rr          # sets the CUDA allocator option before torch touches the GPU (as the workers)\n    from pork import jp1 as P, jp1_shared_prefix as SP, kv_cached_backend as kv, pork_t_v1 as pt\n    from pork.ct1_forks_replacement import generate_batch\n    import torch\n    _t, tok, model = rr.load_model(model_dir, None, "cuda", dict(P.MODELS["1b"], dtype="float32"))\n    assert str(model.dtype) == "torch.float32"\n    B = pt.backend("torch", "cuda")\n    ids = tuple(kv.chat_prompt_ids(tok, P.CHECK_PROMPTS[0][1]))\n    pre = (ids * (length // len(ids) + 1))[:length - steps]          # harmless filler; prefix + steps = length\n    streams = [P.stream("free", 0, 0, j) for j in range(P.K)]\n    wait("loaded")\n    torch.cuda.reset_peak_memory_stats()\n    t0 = time.time()\n    generate_batch(B, SP.torch_shared_prefix_batch(model, torch, "cuda")(), pre, streams, (-1,), P.TEMPERATURE, steps)\n    torch.cuda.synchronize()\n    res.update({"ok": True, "seconds": time.time() - t0, "prefix_tokens": len(pre), "steps": steps,\n                "peak_allocated_gb": torch.cuda.max_memory_allocated() / 1e9,\n                "peak_reserved_gb": torch.cuda.max_memory_reserved() / 1e9})\nexcept BaseException as e:\n    res["error"] = type(e).__name__\n    res["error_tail"] = traceback.format_exc().strip().splitlines()[-1][:300]\nfinally:\n    try:\n        wait("done")\n        import torch as _tt\n        free_b, total_b = _tt.cuda.mem_get_info()\n        res["device_used_gb_while_both_hold"] = (total_b - free_b) / 1e9\n    except BaseException as e2:\n        res["barrier_error"] = type(e2).__name__\n    Path(out).write_text(json.dumps(res))\n', encoding="utf-8")
BAR = Path("/content/jp1_probe_barrier")
if BAR.exists():
    shutil.rmtree(BAR)
BAR.mkdir()
PROBE_STEPS = 16
procs = {}
for tag in ("w0", "w1"):
    procs[tag] = subprocess.Popen([sys.executable, str(PROBE_PY), str(REPO), str(MODEL_DIR), str(LONGEST), str(PROBE_STEPS),
                                   f"/content/jp1_probe_{tag}.json", tag, str(BAR)],
                                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for tag, pr in procs.items():
    pr.wait(timeout=1800)
TWO = {}
for tag, pr in procs.items():
    p = Path(f"/content/jp1_probe_{tag}.json")
    TWO[tag] = json.loads(p.read_text()) if p.exists() else {"tag": tag, "ok": False, "error": "no output"}
    TWO[tag]["returncode"] = pr.returncode
TWO_OK = all(v.get("ok") and v["returncode"] == 0 for v in TWO.values())
write_json("PREFIX_TWO_WORKERS.json", {"longest_sequence": LONGEST, "both_ok": TWO_OK, "workers": TWO})
print(json.dumps({"both_ok": TWO_OK, **{k: {kk: (round(vv, 2) if isinstance(vv, float) else vv) for kk, vv in v.items()}
                                          for k, v in TWO.items()}}, indent=1))

In [ ]:
# Cell 4 — in the notebook (one process): E2 / G1 / B1, the new step against the old one, timing smoke, memory.
import gc
import torch as torch_mod
from pork import jp1_runner as rr
from pork import jp1_shared_prefix as SP
from pork import pork_t_v1 as pt
from pork import quality_pork_q8t as q8
from pork.ct1_runner import torch_fns
from pork.dp1_forks_runner import generate_base
from pork.ct1_forks_replacement import generate_batch
Bt = pt.backend("torch", "cuda")
FLAGS = {"cuda_matmul_allow_tf32": bool(torch_mod.backends.cuda.matmul.allow_tf32),
         "float32_matmul_precision": torch_mod.get_float32_matmul_precision()}
assert FLAGS["cuda_matmul_allow_tf32"] is False and FLAGS["float32_matmul_precision"] == "highest", FLAGS
_t, TOKC, MODEL = rr.load_model(str(MODEL_DIR), None, "cuda", dict(P.MODELS["1b"], dtype="float32"))
assert str(MODEL.dtype) == "torch.float32" and MODEL.config._attn_implementation == P.MODELS["1b"]["attn_implementation"]
EOS = kv.eos_ids(TOKC, MODEL)
single, batch_old = torch_fns(MODEL, torch_mod, "cuda")
batch_new = SP.torch_shared_prefix_batch(MODEL, torch_mod, "cuda")

# (a) the gate's E2 / G1 (unchanged code, float32) and B1 with both batched steps
def b1(batch_factory):
    ids = kv.chat_prompt_ids(TOKC, P.CHECK_PROMPTS[0][1])
    _bs, _ss = batch_factory(), single()
    bp, sp, bn, sn = None, None, [tuple(ids)] * P.K, tuple(ids)
    agree = 0
    for _ in range(32):
        bl, bp = _bs(bn, bp); sl, sp = _ss(sn, sp)
        t_ = int(sl.argmax()); agree += int(int(bl[0].argmax()) == t_)
        bn, sn = [(t_,)] * P.K, (t_,)
    return {"agree_steps": agree, "identical": agree >= P.B1_MIN_AGREE}
e2, g1 = [], []
for name, text in P.CHECK_PROMPTS:
    ids = kv.chat_prompt_ids(TOKC, text)
    step = single()
    seq, past, new, worst, agree = list(ids), None, tuple(ids), 0.0, True
    for _ in range(P.E2_STEPS):
        a, past = step(new, past)
        with torch_mod.no_grad():
            bfull = MODEL(input_ids=torch_mod.tensor([seq], device="cuda"), use_cache=False).logits[0, -1].float()
        worst = max(worst, float((a - bfull).abs().max()))
        ta, tb = int(a.argmax()), int(bfull.argmax())
        agree = agree and ta == tb
        seq.append(tb); new = (tb,)
    e2.append({"prompt": name, "max_abs_logit_diff": worst, "argmax_agree": agree})
    ref_tokens, seq = [], list(ids)
    for _ in range(P.G1_STEPS):
        with torch_mod.no_grad():
            tok_ = q8._greedy([float(v) for v in MODEL(input_ids=torch_mod.tensor([seq], device="cuda"),
                                                        use_cache=False).logits[0, -1].float().tolist()])
        ref_tokens.append(tok_); seq.append(tok_)
        if tok_ in set(EOS):
            break
    base = generate_base(Bt, single(), ids, EOS, len(ref_tokens))["token_ids"]
    g1.append({"prompt": name, "steps": len(ref_tokens), "identical": base == ref_tokens})
GATE = {"E2": e2, "G1": g1, "B1_old_step": b1(batch_old), "B1_new_step": b1(batch_new)}
GATE["pass_with_new_step"] = (all(r["argmax_agree"] for r in e2) and all(r["identical"] for r in g1)
                              and GATE["B1_new_step"]["identical"])
print(json.dumps({"E2": [r["argmax_agree"] for r in e2], "G1": [r["identical"] for r in g1],
                  "B1_old": GATE["B1_old_step"]["agree_steps"], "B1_new": GATE["B1_new_step"]["agree_steps"]}))

# (b) the new step against the old one: same prefixes, same streams -> the same 32 samples?
EQ_STEPS = 48
def first_logits(factory, pre):
    step = factory()
    lg, _ = step([tuple(pre)] * P.K, None)
    return lg
EQ = []
for name, text in P.CHECK_PROMPTS:
    ids = tuple(kv.chat_prompt_ids(TOKC, text))
    path = generate_base(Bt, single(), ids, EOS, 300)["token_ids"]
    for ci, cut in enumerate(c for c in (0, 64, 150, 280) if c <= len(path)):
        pre = ids + tuple(path[:cut])
        streams = [P.stream("free", 1, ci, j) for j in range(P.K)]
        lo, ln = first_logits(batch_old, pre), first_logits(batch_new, pre)
        a = generate_batch(Bt, batch_old(), pre, streams, EOS, P.TEMPERATURE, EQ_STEPS)
        b = generate_batch(Bt, batch_new(), pre, streams, EOS, P.TEMPERATURE, EQ_STEPS)
        EQ.append({"prompt": name, "cut": cut, "prefix_tokens": len(pre),
                   "rows_identical": sum(x == y for x, y in zip(a, b)), "rows": P.K,
                   "first_step_max_abs_logit_diff": float((lo - ln).abs().max()),
                   "old_rows_all_equal_at_first_step": bool((lo == lo[0:1]).all()),
                   "sha256_old": hashlib.sha256(json.dumps([o["token_ids"] for o in a]).encode()).hexdigest(),
                   "sha256_new": hashlib.sha256(json.dumps([o["token_ids"] for o in b]).encode()).hexdigest()})
        del lo, ln, a, b
EQ_ALL = all(r["rows_identical"] == r["rows"] for r in EQ)
print(json.dumps({"rows_identical": [(r["prompt"], r["cut"], r["rows_identical"]) for r in EQ],
                  "max_first_step_logit_diff": max(r["first_step_max_abs_logit_diff"] for r in EQ)}))

# (c) G0 cell 7's timing smoke with the new step (same formula), and one-process memory at the longest sequence
total_gb = torch_mod.cuda.get_device_properties(0).total_memory / 1e9
WORKERS = max(1, min(rr.MAX_WORKERS, kv.physical_cores(), int(total_gb // 10)))
ids = kv.chat_prompt_ids(TOKC, P.CHECK_PROMPTS[0][1])
no_eos = (-1,)
generate_base(Bt, single(), ids, no_eos, 8)
n = P.TIMING_SMOKE_STEPS
per = {}
t0 = time.time(); g = generate_base(Bt, single(), ids, no_eos, n); per["BASE"] = (time.time() - t0) / n
streams = [P.stream("free", 0, 0, j) for j in range(P.K)]
generate_batch(Bt, batch_new(), ids, streams, no_eos, P.TEMPERATURE, 4)
t0 = time.time(); generate_batch(Bt, batch_new(), ids, streams, no_eos, P.TEMPERATURE, n); per["BATCH32"] = (time.time() - t0) / n
long_ids = tuple(ids) + tuple(g["token_ids"]) * 12
t0 = time.time(); generate_batch(Bt, batch_new(), long_ids, streams, no_eos, P.TEMPERATURE, 1); per["PREFILL"] = time.time() - t0
KPP = 3 + 1 / P.FREE_EVERY
def per_item(Lb):
    rest = [Lb - t for t in range(0, Lb, P.STRIDE)]
    return Lb * per["BASE"] + sum(rest) * KPP * per["BATCH32"] + len(rest) * KPP * per["PREFILL"]
NB = {}
for L in ("ja", "en"):
    hours = len(P.HALVES["A"]) * per_item(MEDIAN[L]) / WORKERS / 3600
    NB[L] = {"median_base_tokens": MEDIAN[L], "typical_hours_per_notebook": hours,
             "calibrated_hours_per_notebook": hours * P.TIMING_FACTOR}
main_cu = 2 * sum(v["calibrated_hours_per_notebook"] for v in NB.values()) * P.L4_CU_PER_HOUR
gc.collect(); torch_mod.cuda.empty_cache(); torch_mod.cuda.reset_peak_memory_stats()
pre = (tuple(ids) * (LONGEST // len(ids) + 1))[:LONGEST - 16]
generate_batch(Bt, batch_new(), pre, streams, no_eos, P.TEMPERATURE, 16)
ONE = {"sequence_tokens": LONGEST, "peak_allocated_gb": torch_mod.cuda.max_memory_allocated() / 1e9,
       "peak_reserved_gb": torch_mod.cuda.max_memory_reserved() / 1e9}
TIMING = {"workers_g0_rule": WORKERS, "per_step_s": per, "per_notebook": NB, "main_run_cu_4_notebooks": main_cu,
          "per_notebook_ok": all(v["calibrated_hours_per_notebook"] <= P.TIMING_STOP_HOURS for v in NB.values()),
          "gpu_total_gb": total_gb, "one_process_longest": ONE}
RES = {"precision_flags": FLAGS, "gate": GATE, "equivalence": EQ, "equivalence_all_rows_identical": EQ_ALL, "timing": TIMING}
write_json("PREFIX_IN_NOTEBOOK.json", RES)
print(json.dumps({"calibrated_hours_per_notebook": {L: round(v["calibrated_hours_per_notebook"], 2) for L, v in NB.items()},
                  "main_run_cu": round(main_cu, 1), "per_step_s": {k: round(v, 4) for k, v in per.items()},
                  "one_process_peak_reserved_gb": round(ONE["peak_reserved_gb"], 2)}, indent=1))

In [ ]:
# Cell 5 — summary (numbers only, under 20 KB), then notify and disconnect.
SUMMARY = {"schema": "jp1_prefix_check_v1", "head": EXPECTED_HEAD, "gpu": GPU_NAME, "checks": CHECKS,
           "finished_jst": dt.datetime.now(JST).isoformat(timespec="seconds"), "elapsed_hours": (time.time() - STARTED) / 3600,
           "prompt_tokens": PROMPT_LEN, "longest_sequence": LONGEST, "g0_run1_median_base_tokens": MEDIAN,
           "two_workers": {"both_ok": TWO_OK, "workers": TWO}, "in_notebook": RES}
sha_ = write_json("PREFIX_SUMMARY.json", SUMMARY)
print("PREFIX_SUMMARY.json sha256:", sha_, "| bytes:", (D / "PREFIX_SUMMARY.json").stat().st_size)
notify("先頭1回の確認 終わりました（結果は Drive の jp1/PREFIXCHECK）", tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()